# PHẦN 0: KHỞI TẠO CẤU HÌNH TOÀN CỤC (GLOBAL CONFIGURATION)
Mọi tệp tin lập trình trong dự án phải dùng chung một tập hợp các biến môi trường để đảm bảo tính nhất quán.
**Lưu ý:** Pipeline thiết lập tuân thủ quy tắc Chống Rò Rỉ Time-Leak và Chống Rò Rỉ Cold-Start. Chỉ tập Train được tham gia chạy K-core và tạo Ánh Xạ. Data xử lý dạng Chunking theo Sequential File I/O để triệt tiêu OOM.

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
import gzip
import random
from collections import Counter

# --- Cấu hình Toàn cục ---
IS_SAMPLE = False  # Công tắc định tuyến luồng dữ liệu.
SAMPLE_USER_COUNT = 1000000  # Giới hạn số lượng người dùng khi IS_SAMPLE = True
CHUNK_SIZE = 500000  # Kích thước khối dữ liệu xử lý trên RAM trong mỗi chu kỳ
TRAIN_RATIO = 0.8  # Tỷ lệ cắt mốc thời gian tự động (VD 80% Data Cũ làm Train, 20% Data Mới làm Test)
NEGATIVE_RATIO = 4  # Tỷ lệ lấy mẫu âm (1 mẫu dương : 4 mẫu âm)

print("- Global Configuration Loaded -")
print(f"IS_SAMPLE: {IS_SAMPLE}")
print(f"SAMPLE_USER_COUNT: {SAMPLE_USER_COUNT}")
print(f"CHUNK_SIZE: {CHUNK_SIZE}")
print(f"TRAIN_RATIO: {TRAIN_RATIO}")
print(f"NEGATIVE_RATIO: {NEGATIVE_RATIO}")

- Global Configuration Loaded -
IS_SAMPLE: True
SAMPLE_USER_COUNT: 1000000
CHUNK_SIZE: 500000
TRAIN_RATIO: 0.8
NEGATIVE_RATIO: 4


# PHẦN 1: TIỀN XỬ LÝ CHUẨN MỰC BẢO TOÀN DỮ LIỆU

In [3]:
# Khai báo các đường dẫn thư mục
RAW_REVIEW_PATH = '../data/raw/Clothing_Shoes_and_Jewelry.jsonl.gz'
RAW_META_PATH = '../data/raw/meta_Clothing_Shoes_and_Jewelry.jsonl.gz'

PROCESSED_DATA_DIR = '../data/processed'
INTERIM_DATA_DIR = '../data/interim'
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(INTERIM_DATA_DIR, exist_ok=True)

INTERIM_REVIEW_PATH = os.path.join(INTERIM_DATA_DIR, 'interim_reviews.csv')
INTERIM_TRAIN_PATH = os.path.join(INTERIM_DATA_DIR, 'interim_train.csv')
INTERIM_TEST_PATH = os.path.join(INTERIM_DATA_DIR, 'interim_test.csv')

### BƯỚC 1: Đọc Dữ liệu, Gán Nhãn, Lấy Mẫu (Chuẩn Chunking)

In [4]:
def extract_and_sample_reviews(input_path, output_path, is_sample=IS_SAMPLE, sample_size=SAMPLE_USER_COUNT, chunk_size=CHUNK_SIZE):
    """
    Đọc luồng dữ liệu bằng chuẩn json và gzip nguyên thủy để triệt tiêu Pandas MemoryError.
    Trích xuất cột, gán nhãn, và Sample User trực tiếp trên Chunk.
    """
    import gzip, json, numpy as np, pandas as pd
    review_cols = ['user_id', 'parent_asin', 'timestamp', 'rating']
    sampled_users = None
    
    if is_sample:
        print("Dang quet unique users...")
        unique_users = set()
        with gzip.open(input_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line.strip())
                    if 'user_id' in data:
                        unique_users.add(data['user_id'])
                except:
                    pass
        
        unique_users = list(unique_users)
        if len(unique_users) > sample_size:
            sampled_users = set(np.random.choice(unique_users, sample_size, replace=False))
        else:
            sampled_users = set(unique_users)
            
    print("Dang doc chunk va ghi...")
    chunk_data = []
    first_chunk = True
    
    with gzip.open(input_path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
            except:
                continue
            
            row = {}
            for col in review_cols:
                if col in data:
                    row[col] = data[col]
                    
            if 'rating' in data:
                row['label'] = 1 if data['rating'] >= 4 else 0
                
            if is_sample and sampled_users is not None:
                if row.get('user_id') not in sampled_users:
                    continue
                    
            if row:
                chunk_data.append(row)
            
            if len(chunk_data) >= chunk_size:
                df_chunk = pd.DataFrame(chunk_data)
                df_chunk.to_csv(output_path, mode='w' if first_chunk else 'a', index=False, header=first_chunk)
                first_chunk = False
                chunk_data = []
                
    if chunk_data:
        df_chunk = pd.DataFrame(chunk_data)
        df_chunk.to_csv(output_path, mode='w' if first_chunk else 'a', index=False, header=first_chunk)


### BƯỚC 2: Cắt Chia Train/Test Xuyên Không Gian Bằng Mốc Động (Dynamic Cut-Off)

In [5]:
def find_dynamic_cutoff_date_chunked(file_path, chunk_size=CHUNK_SIZE, ratio=TRAIN_RATIO):
    """
    Hệ thống quét mốc thời gian cắt Train/Test tự động mà không gây OOM.
    Tuyệt đối chỉ nạp cột timestamp, phớt lờ mọi dữ liệu khác.
    """
    print("Đang phân tích dòng thời gian để tìm mốc Cut-off tự động...")
    timestamps = []
    
    # Chỉ định usecols=['timestamp'] để ép Pandas không đọc văn bản rác
    reader = pd.read_csv(file_path, usecols=['timestamp'], chunksize=chunk_size)
    
    for chunk in reader:
        # Lưu dưới dạng numpy array để tối ưu tối đa bộ nhớ
        timestamps.append(chunk['timestamp'].values)
        
    # Gộp tất cả các chunk thành 1 mảng 1 chiều duy nhất
    all_timestamps = np.concatenate(timestamps)
    
    # Sắp xếp mảng số nguyên (nhanh và nhẹ hơn hàng nghìn lần so với sort DataFrame)
    all_timestamps.sort()
    
    # Xác định vị trí Index ở tỷ lệ dựa theo cấu hình (VD: 80%)
    cut_off_index = int(len(all_timestamps) * ratio)
    cut_off_val = all_timestamps[cut_off_index]
    
    # Giải phóng mảng khổng lồ khỏi RAM ngay lập tức
    del all_timestamps
    del timestamps
    
    # Chuyển đổi mili-giây sang định dạng Datetime
    cutoff_date = pd.to_datetime(cut_off_val, unit='ms')
    
    return cutoff_date.strftime('%Y-%m-%d') # Trả về chuỗi ngày chuẩn xác

def time_split_chunked(input_path, train_output, test_output, cutoff_date_str, chunk_size=CHUNK_SIZE):
    """
    Bổ dọc tập dữ liệu theo Time Cutoff. Tách riêng Data trước Date để Train, Data sau Date làm Test.
    """
    cutoff = pd.to_datetime(cutoff_date_str)
    reader = pd.read_csv(input_path, chunksize=chunk_size)
    
    first_train = True
    first_test = True
    for chunk in reader:
        chunk['review_time'] = pd.to_datetime(chunk['timestamp'], unit='ms')
        
        train_chunk = chunk[chunk['review_time'] < cutoff]
        test_chunk = chunk[chunk['review_time'] >= cutoff]
        
        # Trút xuất trực tiếp thành 2 file vật lý tách biệt
        if not train_chunk.empty:
            train_chunk.to_csv(train_output, mode='w' if first_train else 'a', index=False, header=first_train)
            first_train = False
        if not test_chunk.empty:
            test_chunk.to_csv(test_output, mode='w' if first_test else 'a', index=False, header=first_test)
            first_test = False

### BƯỚC 3: Lọc K-Core Chỉ Thiết Lập Lên Vùng TRAIN


In [ ]:
def process_kcore_only_on_train(train_file_path, k_core=2, chunk_size=CHUNK_SIZE):
    """
    Tiến hành vòng lặp đếm node cắt K-Core chỉ duy nhất trong Train File.
    Chính thức loại bỏ Rò Rỉ Tương Lai (Time-Leakage) và Khử Rò Rỉ Cold-Start.
    """
    iteration = 1
    while True:
        user_counts = Counter()
        item_counts = Counter()
        
        reader = pd.read_csv(train_file_path, chunksize=chunk_size)
        for chunk in reader:
            user_counts.update(chunk['user_id'])
            item_counts.update(chunk['parent_asin'])
            
        valid_users = set(k for k, v in user_counts.items() if v >= k_core)
        valid_items = set(k for k, v in item_counts.items() if v >= k_core)
        
        if len(valid_users) == len(user_counts) and len(valid_items) == len(item_counts):
            print(f"K-core hội tụ trên Tập Train tại vòng lặp {iteration}.")
            break

        # Ghi đè file Train bằng Train-đã-lọc 
        temp_file = train_file_path + ".temp"
        reader = pd.read_csv(train_file_path, chunksize=chunk_size)
        first_chunk = True
        for chunk in reader:
            filtered_chunk = chunk[chunk['user_id'].isin(valid_users) & chunk['parent_asin'].isin(valid_items)]
            if not filtered_chunk.empty:
                filtered_chunk.to_csv(temp_file, mode='w' if first_chunk else 'a', index=False, header=first_chunk)
                first_chunk = False
                
        os.replace(temp_file, train_file_path)
        iteration += 1

    # Trả về Danh Sách Định Danh Hợp Lệ Của Train (Dùng làm bộ từ điển ánh xạ)
    return valid_users, valid_items

### BƯỚC 4: Mapping ID Tập Trung Trải Từ Train Xuống Test

In [7]:
def process_meta_chunked(meta_path, valid_item_ids, output_path, chunk_size=CHUNK_SIZE):
    """
    Giữ lại toàn bộ Thông Tin Meta đối với TẤT CẢ TÀI SẢN tham chiến (Train + Test)
    Đổi sang gzip + json thuần để chống MemoryError
    """
    import gzip, json, pandas as pd
    meta_cols = ['parent_asin', 'price', 'main_category', 'store', 'average_rating', 'rating_number', 'title', 'images']
    valid_items_set = set(valid_item_ids)
    
    chunk_data = []
    first_chunk = True
    
    with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line.strip())
            except:
                continue
                
            if data.get('parent_asin') in valid_items_set:
                row = {col: data.get(col) for col in meta_cols if col in data}
                if row:
                    chunk_data.append(row)
                    
            if len(chunk_data) >= chunk_size:
                df_chunk = pd.DataFrame(chunk_data)
                df_chunk.to_csv(output_path, mode='w' if first_chunk else 'a', index=False, header=first_chunk)
                first_chunk = False
                chunk_data = []
                
    if chunk_data:
        df_chunk = pd.DataFrame(chunk_data)
        df_chunk.to_csv(output_path, mode='w' if first_chunk else 'a', index=False, header=first_chunk)


### BƯỚC 5: Xử lý Tệp Meta

In [8]:
def process_meta_chunked(meta_path, valid_item_ids, output_path, chunk_size=CHUNK_SIZE):
    """
    Giữ lại toàn bộ Thông Tin Meta đối với TẤT CẢ TÀI SẢN tham chiến (Train + Test)
    """
    meta_cols = ['parent_asin', 'price', 'main_category', 'store', 'average_rating', 'rating_number', 'title', 'images']
    reader = pd.read_json(meta_path, lines=True, chunksize=chunk_size, compression='gzip')
    valid_items_set = set(valid_item_ids)
    
    first_chunk = True
    for chunk in reader:
        filtered_chunk = chunk[chunk['parent_asin'].isin(valid_items_set)]
        filtered_chunk = filtered_chunk[[col for col in meta_cols if col in filtered_chunk.columns]]
        
        if not filtered_chunk.empty:
            filtered_chunk.to_csv(output_path, mode='w' if first_chunk else 'a', index=False, header=first_chunk)
            first_chunk = False

### EXECUTION BLOCK

In [9]:
print("BƯỚC 1/5: Đọc, Gán Nhãn và Lấy Mẫu (Chunked)...")
extract_and_sample_reviews(RAW_REVIEW_PATH, INTERIM_REVIEW_PATH)

print("BƯỚC 2/5: Bổ Đôi Lưu Lượng Bằng Time Cut-Off Tự Động...")
DYNAMIC_CUT_OFF = find_dynamic_cutoff_date_chunked(INTERIM_REVIEW_PATH, ratio=TRAIN_RATIO)
print(f"Mốc chia Train/Test tự động theo ({int(TRAIN_RATIO*100)}/{100-int(TRAIN_RATIO*100)}): {DYNAMIC_CUT_OFF}")
time_split_chunked(INTERIM_REVIEW_PATH, INTERIM_TRAIN_PATH, INTERIM_TEST_PATH, cutoff_date_str=DYNAMIC_CUT_OFF)

print("BƯỚC 3/5: Áp Dụng Thuật Toán K-Core DUY NHẤT LÊN TẬP TRAIN...")
train_valid_users, train_valid_items = process_kcore_only_on_train(INTERIM_TRAIN_PATH, k_core=2)
print(f"-> K-Core hoàn thiện. Train Pool giữ {len(train_valid_users)} Users & {len(train_valid_items)} Items.")

print("BƯỚC 4/5: Map Cơ Sở Dữ Liệu...")
TRAIN_OUT_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.csv')
TEST_OUT_PATH = os.path.join(PROCESSED_DATA_DIR, 'test_interactions.csv')
n_users, n_items, test_items_set = map_ids_chunked(
    train_input=INTERIM_TRAIN_PATH, 
    test_input=INTERIM_TEST_PATH, 
    train_output=TRAIN_OUT_PATH, 
    test_output=TEST_OUT_PATH, 
    train_users=train_valid_users, 
    train_items=train_valid_items
)

print("BƯỚC 5/5: Lọc Dữ Liệu Siêu Cấu Trúc (Meta)...")
# Quy chuẩn hợp nhất (Union) để đảm bảo Cold-start Items của Test cũng được sở hữu Meta mô tả tính năng
all_valid_items_for_meta = set(train_valid_items).union(test_items_set)
META_OUT_PATH = os.path.join(PROCESSED_DATA_DIR, 'filtered_metadata.csv')
process_meta_chunked(RAW_META_PATH, all_valid_items_for_meta, META_OUT_PATH)

print(f"Hoàn tất toàn bộ chu trình vạch tuyến! Số Map User (N): {n_users}, Số Map Item (N): {n_items}")
print(f"Tổng số khối lượng Meta Data hợp lệ được cấp quyền ra CSV: {len(all_valid_items_for_meta)}")

BƯỚC 1/5: Đọc, Gán Nhãn và Lấy Mẫu (Chunked)...
BƯỚC 2/5: Bổ Đôi Lưu Lượng Bằng Time Cut-Off Tự Động...
Đang phân tích dòng thời gian để tìm mốc Cut-off tự động...
Mốc chia Train/Test tự động theo (80/20): 2021-12-30
BƯỚC 3/5: Áp Dụng Thuật Toán K-Core DUY NHẤT LÊN TẬP TRAIN...
K-core hội tụ trên Tập Train tại vòng lặp 32.
-> K-Core hoàn thiện. Train Pool giữ 21152 Users & 13794 Items.
BƯỚC 4/5: Map Cơ Sở Dữ Liệu...
BƯỚC 5/5: Lọc Dữ Liệu Siêu Cấu Trúc (Meta)...
Hoàn tất toàn bộ chu trình vạch tuyến! Số Map User (N): 21152, Số Map Item (N): 13794
Tổng số khối lượng Meta Data hợp lệ được cấp quyền ra CSV: 240190


In [2]:
df_test = pd.read_csv('../data/processed/test_interactions.csv')
cold_start_ratio = (df_test['mapped_item_id'] == 0).mean() * 100
print(f"Tỷ lệ sản phẩm bị xóa mất ID ở tập Test: {cold_start_ratio:.2f}%")

Tỷ lệ sản phẩm bị xóa mất ID ở tập Test: 77.00%
